## 2.0 Initial Setup

In [1]:
import os
import sys

In [4]:
current_dir = os.getcwd()

if current_dir.endswith('report'):
    os.chdir('..') # Change working directory for file paths (like checkpoints/ and data/)
    sys.path.append(os.getcwd()) # Add root to Python path for module imports (like models/ and losses/)
    print(f"Working directory changed to project root: {os.getcwd()}")
else:
    print(f"Working directory is already: {current_dir}")

Working directory is already: c:\Users\Vishnu\Desktop\IITM\Sem-2\DL\Assignments\Assignment-02


## 2.1 The Regularization Effect of Dropout

In [ ]:
import torch
import wandb
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np

from models.classification import VGG11Classifier

In [ ]:
# 1. Initialize W&B
wandb.init(project="DA6401_Assignment_02", name="report_visualizations", job_type="analysis")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Define Validation Transforms (from your inference script)
vgg_mean = (0.485, 0.456, 0.406)
vgg_std = (0.229, 0.224, 0.225)
val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=vgg_mean, std=vgg_std),
    ToTensorV2()
])

# 3. Load a single image for testing (Pick any image from your dataset)
image_path = "data/images/Abyssinian_1.jpg"
raw_pil = Image.open(image_path).convert("RGB")
raw_image = np.array(raw_pil)
transformed = val_transforms(image=raw_image)
input_tensor = transformed["image"].unsqueeze(0).to(device)

# 4. Instantiate both models
model_bn = VGG11Classifier(use_batchnorm=True, dropout_p=0.5).to(device)
model_nobn = VGG11Classifier(use_batchnorm=False, dropout_p=0.5).to(device)

# 5. Load Checkpoints
# Note: Ensure you renamed the no-BN checkpoint so it wasn't overwritten
model_bn.load_state_dict(torch.load("checkpoints/classifier.pth", map_location=device)["state_dict"])
model_nobn.load_state_dict(torch.load("checkpoints/classifier_no_bn.pth", map_location=device)["state_dict"])

# 6. CRITICAL: Set to eval mode
model_bn.eval()
model_nobn.eval()

print("Setup Complete. Models loaded and set to eval mode.")

In [ ]:
# Dictionary to hold our captured activations
activation_cache = {}

# Define the hook function
def get_activation(name):
    def hook(model, input, output):
        # We detach it from the graph and move to CPU immediately
        activation_cache[name] = output.detach().cpu()
    return hook

# Target the 3rd Conv layer (Block 3, item 0 is the Conv2d layer)
hook_bn = model_bn.encoder.block3[0].register_forward_hook(get_activation('with_bn'))
hook_nobn = model_nobn.encoder.block3[0].register_forward_hook(get_activation('no_bn'))

# Pass the image through both models
with torch.no_grad():
    _ = model_bn(input_tensor)
    _ = model_nobn(input_tensor)

# Flatten the tensors for histogram plotting
act_bn_flat = activation_cache['with_bn'].flatten().numpy()
act_nobn_flat = activation_cache['no_bn'].flatten().numpy()

# Log to Weights & Biases
wandb.log({
    "2.1_Activations/With_BatchNorm": wandb.Histogram(act_bn_flat),
    "2.1_Activations/Without_BatchNorm": wandb.Histogram(act_nobn_flat)
})

# Remove the hooks so they don't interfere with later cells
hook_bn.remove()
hook_nobn.remove()

print("Activations logged to W&B successfully!")